# Data Cleaning — Explanation
### A companion lecture to `Data Cleaning.ipynb`

**Student:** Mathonsi Mphikeleli Mbongiseni (28574249) · UNISA MCom Quantitative Management

This notebook does not run the research pipeline. It explains it. Every section below corresponds to a numbered section in `Data Cleaning.ipynb`, in the same order, and walks through the reasoning behind each step: what the code does, why that step exists, and how to read the numbers it produced. It assumes no prior background in programming, statistics, or finance — every term is defined the first time it appears.

A short piece of code from the technical notebook is quoted at the start of each section so the explanation has something concrete to point at. The quoted code is never re-run here; the actual execution, and its real output, lives in `Data Cleaning.ipynb`. Where a number is quoted below (a violation count, a p-value, a date), it is the actual value that notebook produced — not an illustrative placeholder.

## Why does a research project need a "data cleaning" stage at all?

Before any statistic can be trusted, the numbers that feed it have to be trusted first. This sounds obvious, but it is the single most common source of wrong conclusions in applied data analysis, captured by the old saying **garbage in, garbage out**: no amount of clever modelling later in a project can undo a mistake sitting quietly in the raw data at the start of it.

In this dissertation the raw material is *daily share prices* for ten financial instruments — eight individual companies and two market indices — stretching back to January 2010. Data of this kind, pulled from a live financial data provider, can go wrong in several specific ways:

- a price can be **missing** for a day (a data outage, a public holiday handled inconsistently, a   company not yet listed on the exchange);
- the same trading day can appear **twice** in the file, or the dates can arrive out of **order**;
- a price can be **zero, negative, or absurdly large** because of a data-entry fault or an   unadjusted corporate action (a stock split, for example, can make a price look like it fell 90%   overnight when nothing economically happened);
- a price move can be **genuinely extreme** rather than an error — a real crash or a real rally —   and the research design has to decide, deliberately, how such days should be treated rather than   discovering the question by accident later on.

Every one of these problems, if left unnoticed, would flow straight into the Value-at-Risk (VaR) estimates that are the whole point of this dissertation. A VaR figure is a statement about how bad a loss could plausibly get; if the historical data used to compute it is quietly corrupted, the resulting risk estimate is not just imprecise, it is actively misleading — exactly the opposite of what a risk model exists to prevent. This is why the cleaning stage is treated as its own notebook, given its own careful checks, and never skipped.

## 1. Setup

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
```

Every Python script that does data analysis starts by importing **libraries** — pre-written, tested collections of code that save the author from reinventing basic tools. Four libraries do almost all of the work in this project:

- **pandas** provides the `DataFrame`, a spreadsheet-like table that lives in memory. Nearly every   object manipulated in these notebooks — prices, returns, statistics — is a pandas `DataFrame` or   its single-column relative, a `Series`.
- **numpy** provides fast numerical arrays and mathematical functions (logarithms, square roots,   and so on). pandas is built on top of numpy.
- **matplotlib** and **seaborn** draw the charts. seaborn is a layer on top of matplotlib with   nicer defaults for statistical plots such as heatmaps.
- **yfinance** is a Python interface to Yahoo Finance, used here purely to download historical   daily prices — it is the only place external data enters the project.
- **scipy.stats** supplies statistical distributions and tests (the Normal distribution, the   z-score calculation, and later the Jarque-Bera and Student-t machinery used elsewhere).

The setup cell also fixes a handful of **constants** at the top of the notebook — the list of tickers, a lookup from ticker to full company name, the sample start and end dates, and a colour palette for plotting. Defining these once, in capital letters, at the top of the file is a deliberate convention (not a Python requirement) that signals "this value is fixed for the whole notebook and should be changed in exactly one place if it ever needs to change." It also means every later section refers to the same list of ten assets, so there is no risk of one section silently analysing a different set of companies than another.

**A note on vocabulary.** A *ticker* is the short code an exchange uses to identify a listed security — `AAPL` for Apple, `^GSPC` for the S&P 500 index (the caret prefix is Yahoo Finance's convention for indices, which are not directly tradeable instruments themselves). The ten instruments used throughout this dissertation are eight large, liquid US-listed companies spanning technology and banking — Microsoft, Apple, NVIDIA, IBM, Cisco, JPMorgan, Bank of America, and Citigroup — plus two broad market indices, the S&P 500 and the NASDAQ Composite, used later as a point of comparison rather than as portfolio holdings.

## 2. Raw Data

```python
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)
prices = raw['Close'].copy()
```

`yf.download` is the single line that fetches everything: ten years — in fact sixteen — of daily closing prices for all ten tickers in one call. Two choices in that line matter enough to spell out.

**`auto_adjust=True`** asks Yahoo Finance to return *adjusted* closing prices rather than the raw quoted price. Two corporate events otherwise distort a raw price series: a **stock split** (a company turning one share into several, e.g. NVIDIA's repeated splits over this sample period) instantly divides the quoted price without the company being worth any less, and a **dividend payment** mechanically reduces the share price by roughly the dividend amount on the ex-dividend date, again without any change in the value delivered to a shareholder who reinvests it. An adjusted price series retroactively rescales history so that these mechanical jumps disappear, leaving only genuine market price movement. Skipping this step would inject enormous, entirely artificial "crashes" and "rallies" into the return series computed in the next notebook — exactly the kind of silent data fault this whole stage of the project exists to prevent.

**The date range**, 2010-01-01 to 2025-12-31, was chosen to comfortably clear the 2008–09 Global Financial Crisis (so that period does not contaminate the sample with its own distinct dynamics) while still covering more than one and a half decades and several distinct market regimes: the post-crisis recovery, a long bull market, the COVID-19 shock of 2020, and the recent AI-driven rally.

**Reading the output.** The notebook reports **4,023 trading days** across **10 assets**, running from 2010-01-04 to 2025-12-30. Note this is *not* the same as the number of calendar days in the period — roughly 5,840 calendar days sit between those two dates, but a stock exchange is only open on **trading days**: Monday to Friday, minus public holidays. Financial time series analysis conventionally treats a year as having 252 trading days, a number that will resurface constantly from the next notebook onward whenever a daily statistic is *annualised*.

## 3. Index & Duplicate Checks

```python
print("Monotonic index  :", prices.index.is_monotonic_increasing)
print("Duplicate dates  :", prices.index.duplicated().sum())
print("Weekend rows     :", (prices.index.dayofweek >= 5).sum())
```

A financial time series is only meaningful if its dates are in the right order and each date appears exactly once — a return calculation later on works by comparing each row to the *previous* row, so if the previous row is actually from the wrong day, or is a duplicate of the current row, the resulting number is meaningless.

Four cheap checks catch the most common ways this can go wrong:

1. **Monotonic index** — is the sequence of dates strictly increasing, with no date appearing out    of order? `is_monotonic_increasing` returns a single `True`/`False` for the entire index.
2. **Duplicate dates** — does any date appear more than once? `.duplicated().sum()` counts them.
3. **Weekend rows** — stock exchanges are closed on Saturday and Sunday, so any row whose date falls    on a weekend (`dayofweek >= 5`, since pandas numbers Monday as 0 and Sunday as 6) would itself be    suspicious.
4. **Gaps greater than five calendar days** — the difference between consecutive dates should almost    always be one to three days (a normal weekend). A gap wider than five days would usually mean a    public holiday cluster, but it could also mean a chunk of history is silently missing from the    data provider's feed, so it is worth printing and eyeballing rather than assuming.

**Reading the output.** All four checks come back clean: the index is monotonic, there are zero duplicate dates, zero weekend rows, and zero gaps wider than five days. This is a strong, reassuring signal about the quality of the underlying Yahoo Finance feed for these ten highly liquid, heavily-traded instruments — it means none of the more elaborate repair logic later in the notebook (forward-filling, for instance) is being asked to paper over a structurally broken index.

## 4. Missing Values

```python
miss = pd.DataFrame({'count': prices.isnull().sum(), 'pct': ...})
```

A **missing value**, represented in pandas as `NaN` ("Not a Number"), is simply a gap in the data — a cell where no price was recorded. In equity data, missing values typically arise from a company not yet being listed at the start of the sample window, a trading halt, or a data-vendor outage.

`prices.isnull()` produces a table of `True`/`False` values the same shape as the price table, `True` wherever a value is missing; summing that table by column counts the missing values per asset. `first_valid_index()` and `last_valid_index()` report the first and last date each column actually has a real number, which is a quick way to spot a company that only starts partway through the sample.

The missing-value **heatmap** (`sns.heatmap` on the boolean missingness table) is a standard diagnostic in data cleaning: with ten columns and over four thousand rows, no one is going to spot a handful of scattered gaps by scrolling through numbers, but a red smear across a heatmap jumps out immediately. This is the first of several places in the project where a *visualisation* is chosen specifically because the human eye is far better at pattern-spotting across thousands of data points than at reading the same information as a table of numbers.

**Reading the output.** Every single one of the ten series shows **zero missing values**, with `first_valid_index` = 2010-01-04 and `last_valid_index` = 2025-12-30 for all ten — meaning every asset, including NVIDIA (a much smaller, more speculative company at the start of the sample window than it is today), already had a continuously trading, recorded price throughout the entire sixteen-year window. The heatmap is accordingly a uniform, unbroken colour with no red cells. This is a genuinely clean starting point, which is worth stating plainly rather than assuming: the later imputation logic in Section 8 is present as a safety net for a *future* re-run of this pipeline (a longer date range, a different ticker, a data-vendor hiccup), not because this specific dataset needed it.

## 5. Price Sanity Checks

```python
z_neg = (prices <= 0).sum()
chg = prices.pct_change()
ex = chg[t][chg[t].abs() > 0.40]
```

A **sanity check** is a deliberately simple, cheap test applied early, whose only job is to catch an implausible value before it can quietly propagate through every later calculation. Two are run here.

**Zero or negative prices.** A publicly traded company's share price cannot fall to zero or below while it continues to trade — limited liability means the worst a shareholder can do is lose their entire investment, at which point the stock is delisted rather than trading at a negative number. Any zero or negative value in this column would therefore be unambiguous evidence of a data error, not a real market outcome.

**Single-day moves larger than 40%.** `pct_change()` computes the simple percentage change from one row to the next (`(P_t - P_{t-1}) / P_{t-1}`); flagging any daily move whose absolute size exceeds 40% is a generous threshold deliberately set high enough that it should almost never fire on real market data — genuinely large one-day equity moves (an earnings shock, a takeover announcement) are usually in the 10–25% range even in extreme cases. A move beyond 40% most often signals an unadjusted stock split or a ticker/vendor mix-up rather than a real trading day, which is exactly the kind of fault Section 2's `auto_adjust=True` is meant to have already prevented; running this check afterwards is a second, independent line of defence rather than blind trust in one setting.

**Reading the output.** Zero assets show any zero or negative price, and zero single-day moves exceed the 40% threshold across the entire ten-asset, sixteen-year panel. Combined with Sections 3 and 4, this confirms the raw feed is, in practical terms, already clean before any explicit repair logic runs — a reassuring but not entirely unexpected result, since these are ten of the most heavily traded, closely scrutinised instruments on any US exchange, exactly the kind of security for which a mainstream data vendor's coverage is most reliable.

## 6. Outliers in Log Returns

This is the most conceptually important section of the cleaning notebook, and it introduces two ideas — log returns and statistical outlier detection — that recur throughout the rest of the dissertation.

### From prices to returns

A raw price series is not, by itself, the right object to analyse for risk. A price is heavily **trending** (Apple's price is not comparable across 2010 and 2025 in any direct sense) and its scale is arbitrary (NVIDIA traded at a fraction of a dollar per share in early 2010 and hundreds of dollars later in the sample, purely because of stock splits, without that changing anything about the *risk* of holding the stock). What actually matters for both an investor and a risk manager is the **return** — how much value was gained or lost, expressed as a proportion of what was invested.

This dissertation uses the **log return**, defined for a price series $P_t$ as

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

rather than the simple percentage return $(P_t - P_{t-1})/P_{t-1}$. Three properties make the log return the standard choice in quantitative finance:

1. **Time-additivity.** The log return over five days is exactly the *sum* of the five daily log    returns, because $\ln(P_5/P_0) = \ln(P_1/P_0) + \ln(P_2/P_1) + \dots + \ln(P_5/P_4)$. Simple    percentage returns do not add this cleanly across time — they compound multiplicatively instead    — which makes log returns far more convenient for aggregating and modelling over different    horizons.
2. **Symmetry.** A simple return is bounded below by $-100\%$ (a total loss) but unbounded above,    which makes gains and losses of the "same size" look asymmetric. A log return treats a doubling    and a halving of price as equal-and-opposite in size ($\ln 2$ and $-\ln 2$), which better    matches how a continuously-compounding return actually behaves.
3. **Closer to Normal**, at least approximately, over short horizons — an assumption several classical    risk models lean on, even though this dissertation's own tests later show that assumption breaks    down in the tails for every one of these ten assets.

For small daily moves, a log return and a simple return are numerically almost identical (a 1% simple return is a 0.995% log return); the difference only becomes visually noticeable for the very large moves this section is specifically trying to find.

### What is a statistical outlier, and why look for one here?

An **outlier** is an observation that sits unusually far from the bulk of the data. The **z-score** of a return $r$ is $(r - \bar r)/s$, i.e. how many standard deviations $r$ sits away from the sample mean $\bar r$. Under a Normal distribution, a $|z| > 4$ event is vanishingly rare — roughly one in every 15,800 observations. If a return series were genuinely Normal, essentially none of the 4,022 daily returns in each column should exceed that threshold.

A second, **robust** version of the same idea is also computed, based on the **median absolute deviation (MAD)** rather than the mean and standard deviation: `0.6745 * (r - median) / MAD`. The reason both are used side by side is a subtlety worth dwelling on: a small number of extreme observations *inflate the very standard deviation being used to judge them*, which can make a genuine outlier look artificially less extreme (an outlier "hiding itself" inside an inflated denominator). The median and MAD are far less sensitive to a handful of extreme points, so the MAD-based rule tends to flag more observations as unusual — and the executed notebook shows exactly that pattern: for example Citigroup shows 25 events by the z-score rule but 123 by the MAD rule, and similar multi-fold gaps appear for every other asset. That gap is not a bug in either rule; it is direct, quantitative evidence that these return series have **fat tails** — a formal way of saying extreme events happen far more often than a Normal distribution would predict — a stylised fact that the next notebook (the EDA) tests far more rigorously with the Jarque-Bera test.

### Reading the specific numbers

NVIDIA has the single largest logged move in the entire panel: **+26.09% on 2016-11-11**, a genuine, idiosyncratic, earnings-driven rally rather than a market-wide event — a reminder that a "statistical outlier" and a "data error" are not the same thing. By contrast, Bank of America's largest move, **-22.71% on 2011-08-08**, coincides with the 2011 US sovereign credit-rating downgrade panic, and a large fraction of the outliers flagged for *every one* of the ten assets — annotated explicitly in the notebook's output with a `[COVID crash]` tag — cluster in March 2020, the single most extreme, broad-based event in the sample. This clustering in time is itself an important stylised fact previewed here and returned to formally in the EDA notebook under the name **volatility clustering**: extreme days do not scatter randomly through the calendar, they arrive in bursts around identifiable crises.

## 7. Outlier Treatment — and why some outliers must survive

```python
def winsorise(s, lo=0.005, hi=0.995):
    return s.clip(lower=s.quantile(lo), upper=s.quantile(hi))
```

Having *found* unusual returns in Section 6, the natural next question is what, if anything, to do about them. Two options exist in general data cleaning: delete the offending rows entirely, or **winsorise** them — cap any value beyond a chosen percentile at that percentile's value, without removing the observation from the dataset. `s.quantile(0.005)` finds the value below which the smallest 0.5% of observations fall, and `s.clip(lower=..., upper=...)` pulls in any value beyond those bounds to sit exactly on the boundary.

**Why not simply delete extreme returns?** This is the single most important judgement call in the entire cleaning notebook, and it is worth explaining carefully because it runs against a beginner's first instinct ("outliers are bad, remove them"). In most data science applications that instinct is reasonable — a sensor glitch or a typo genuinely does not belong in the dataset. But a Value-at-Risk model exists for exactly one purpose: to describe how bad a plausible loss can get. The single worst days in this sample — the COVID-19 crash of March 2020, the 2011 sovereign-debt panic, the 2008–09 financial crisis just outside this sample's start — are not measurement errors. They are the single most economically important observations in the *entire* dataset for a VaR research project, precisely because they are the days a risk model is being built to anticipate. Deleting them would produce a model that looks well-behaved in a backtest and then fails exactly when it matters, by systematically understating how bad a real crisis can be. This is why the code comment in the technical notebook is explicit: *"extreme events are genuine (COVID, GFC) — retain raw returns for GARCH."*

The compromise adopted here is to keep **two parallel versions** of the return series side by side: the raw, untouched returns (`log_returns_clean.csv`), used for every model in this dissertation that is specifically designed to handle heavy tails (the GARCH family, in particular, is built around modelling exactly this kind of extreme, clustered volatility), and a **winsorised** copy (`log_returns_winsorised.csv`) at the 0.5th/99.5th percentile, offered as an alternative input for any model that could otherwise be unduly distorted by a small number of extreme leverage points — an ordinary least-squares regression, for instance, is well known to be sensitive to exactly this kind of extreme observation. Neither version is presented as strictly "the clean data"; the choice of which to use is left to whichever downstream model needs it, and is documented rather than made silently.

**Reading the output.** Exactly **42 observations are clipped per asset** — and this number is not a coincidence. With 4,022 observations per column and a symmetric 0.5%/99.5% cutoff, exactly 1% of the sample (roughly 40 observations) is expected to be clipped by construction, split between the extreme left and right tails; 42 is that expected count almost exactly. The later side-by-side histogram of NVIDIA and the S&P 500, raw versus winsorised, makes the effect visible directly: the winsorised histogram's tails are pulled in, and its printed excess kurtosis is measurably lower than the raw series — a first, visual preview of the kurtosis statistic that the EDA notebook defines and computes formally for every asset.

## 8. Missing Value Treatment & Final Clean

```python
prices_clean = prices.ffill(limit=2).bfill().dropna(how='any')
```

Three repair techniques are chained together here, each with a distinct role:

- **Forward-fill** (`ffill`) carries the last known valid price forward to fill a gap — if Tuesday's   price is missing, Monday's price is used in its place. The `limit=2` argument caps this at   bridging at most two consecutive missing days; beyond that, the code deliberately refuses to   guess, on the reasoning that a short, one- or two-day gap is plausibly a minor data hiccup, while   a longer gap likely reflects something real (a trading halt, a delisting) that should not be   silently papered over with a stale price.
- **Backward-fill** (`bfill`) catches any gap right at the very start of a series that forward-fill   cannot reach, since there is no earlier valid price to carry forward yet.
- **`dropna(how='any')`** is the final safety net: any row that still contains a missing value after   the two fill steps above is removed entirely, rather than allowed to silently carry a `NaN` into   every later calculation.

**Reading the output.** Because Section 4 already established that this particular dataset has zero missing values, this step is — for this specific run — a no-op: the printed summary shows exactly 4,023 rows both before and after, with **zero rows dropped**. It would be reasonable to ask why the code bothers with this logic at all if it changes nothing here. The answer is **robustness**: this notebook is designed to be re-run — with an extended date range, an additional ticker, or simply a different day's pull from Yahoo Finance that happens to have a gap — without a human needing to notice and hand-patch a new problem each time. Writing defensive code that handles a problem which *might* occur, and verifying explicitly that it did no harm when the problem does not occur, is a core habit of reliable data pipelines.

## 9. Post-Clean Validation

```python
checks = pd.DataFrame({NAMES[t]: {'N obs': ..., 'NaN': ..., 'Zero/neg': ..., 'Dup dates': ...} ...})
print(f"All clean: {checks['NaN'].eq(0).all() and not checks['Zero/neg'].any()}")
```

This section repeats, deliberately, several of the same checks already run in Sections 3 through 5 — but this time against the *final* cleaned table rather than the raw download. This is a standard practice in any data pipeline with more than one processing step: **never assume a cleaning operation achieved what it was supposed to achieve — verify it explicitly, on the output, every time.** It is a cheap check to run and an expensive mistake to skip, since every later notebook in this dissertation reads from the file this step signs off on.

**Reading the output.** All ten assets show 4,023 observations, zero missing values, no zero or negative prices, and no duplicate dates, and the final printed line reads `All clean: True`. This is the formal gate the data must pass before it is trusted enough to be exported and used as the foundation for every subsequent notebook in the project.

## 10. Clean Log Returns

```python
ret_clean = np.log(prices_clean / prices_clean.shift(1)).dropna()
print(ret_clean.describe().rename(columns=NAMES).round(6).to_string())
```

With a fully validated price table in hand, the log return formula introduced in Section 6 is applied for the last time to produce the definitive return series this dissertation builds on. `prices_clean.shift(1)` shifts every value down by one row, so dividing the table by its own shifted version lines up each day's price with the *previous* day's price at every row, exactly matching the $P_t / P_{t-1}$ in the log-return formula; `.dropna()` then removes the single row at the very start of the sample, which has no previous day to compare against.

`describe()` is one of pandas' most-used convenience functions: for each column it reports the **count** (number of observations), **mean** (average daily return), **std** (standard deviation, a measure of how spread out the daily returns are — this is the everyday, working definition of **volatility** used throughout finance), the **minimum** and **maximum** observed values, and the **25th, 50th (median), and 75th percentiles** — respectively, the value below which a quarter, half, and three-quarters of the observations fall. This table is a first, compact snapshot of every asset's behaviour; the EDA notebook that follows this one takes each of these numbers and builds an entire, far more detailed statistical picture around it, adding skewness, kurtosis, and formal hypothesis tests for the properties this table can only hint at (for instance, notice that NVIDIA's standard deviation of 0.0286, or 2.86% per day, dwarfs the S&P 500's 0.0109 — a first glimpse of the far higher volatility of a single, more speculative growth stock relative to a broad, diversified index).

The **normalised price plot** that follows (`prices_clean[t] / prices_clean[t].iloc[0]`) rescales every asset's price history to start at exactly 1.0. This matters because the ten raw price series sit on wildly different scales — NVIDIA around \$0.20 at the very start of the sample versus the S&P 500 around 1,133 — and plotting them together on one chart without rescaling would make nine of the ten lines invisible, flattened near zero by the index's scale. Normalising to a common base of 1 converts every line into "growth of \$1 invested at the start of the period," making relative performance directly, visually comparable across assets that have nothing in common numerically.

## 11. Export

```python
prices_clean.to_csv(DATA_DIR / 'prices_clean.csv')
ret_clean.to_csv(DATA_DIR / 'log_returns_clean.csv')
ret_wins.to_csv(DATA_DIR / 'log_returns_winsorised.csv')
```

The final step writes three files to the project's `data/` folder: the cleaned prices, the clean (unwinsorised) log returns, and the winsorised log returns. This is a **checkpoint pattern**, common to any multi-stage research pipeline: rather than have every downstream notebook re-download and re-clean the data itself — slow, and a risk that two notebooks could silently drift out of sync if the cleaning logic were duplicated — every later notebook in this dissertation reads these three files directly. This has two practical benefits worth naming: it makes each notebook run in seconds rather than minutes, and it guarantees that the *Exploratory Data Analysis* notebook and the *Model Development* notebook are provably working from the exact same, single, validated source of truth.

## Where this leaves the research

By the end of this notebook, the project has moved from an unvalidated download to a dataset that has been checked for ordering, duplication, missingness, implausible prices, and statistically unusual returns — and has made an explicit, justified decision about how to treat the extreme events that were found, rather than either ignoring them or removing them by default. The companion explanation for the next stage, `Exploratory Data Analysis (EDA) - Explanation.ipynb`, picks up exactly where this one leaves off: with `prices_clean.csv` and `log_returns_clean.csv` as its starting point, and the question of *what these ten return series actually look like, statistically* as its subject.